In [2]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd
from glob import glob

# Set option to display all rows (no truncation)
pd.set_option('display.max_rows', None)

# Set option to display all columns (no truncation)
pd.set_option('display.max_columns', None)

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.JwMevPulee/ipykernel_3210897/761733327.py:4: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_

In [2]:

d_readin = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/ncei_events/ncei_ny_events_clean.gpkg")

d_readin.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")

# may take ~40 seconds

In [3]:
# Explore what warnings there are (no need to run)

print(np.unique(d_readin["EVENT_TYPE"]))

eventsofint = ['Blizzard', 'Heavy Rain', 'Heavy Snow', 'Lake-Effect Snow', 'Winter Storm' ,'Winter Weather']

devents = d_readin[d_readin["EVENT_TYPE"].isin(eventsofint)]

# print(len(devents)) # note that this isn't the number of instances to run for inference. Because 1) events elements are a range of time, so need to run multiple 5-min instances within one 'event', so this makes *more* instances, 2) If there an event that spans multiple regions, there is likely overlapping times of the warning. These will end up being duplicate because when we prepare the inference instances to run (below) we run the entire state for every time where there was an active snow squall warning, regardless of where the warning was (cleaner to maintain consistency in inference runs by running every instance statewide)

display(devents.groupby(["EVENT_TYPE"]).count())

['Astronomical Low Tide' 'Blizzard' 'Coastal Flood' 'Cold/Wind Chill'
 'Debris Flow' 'Drought' 'Excessive Heat' 'Extreme Cold/Wind Chill'
 'Flash Flood' 'Flood' 'Frost/Freeze' 'Funnel Cloud' 'Hail' 'Heat'
 'Heavy Rain' 'Heavy Snow' 'High Wind' 'Ice Storm' 'Lake-Effect Snow'
 'Lakeshore Flood' 'Lightning' 'Rip Current' 'Strong Wind'
 'Thunderstorm Wind' 'Tornado' 'Wildfire' 'Winter Storm' 'Winter Weather']


,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration
EVENT_TYPE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Blizzard,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,7,14,14,14,0,0,14,14,14,14,14
Heavy Rain,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,6,6,14,0,0,0,0,0,0,0,0,0,0,0,14,14,14,14,14,14,14,14,14,14,14,11,14,14,0,14,14,14,14,14,14,14
Heavy Snow,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,148,148,185,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,185,157,185,185,185,0,0,185,185,185,185,185
Lake-Effect Snow,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,187,187,211,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,211,57,211,211,211,0,0,211,211,211,211,211
Winter Storm,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,311,311,437,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,437,177,437,437,437,0,0,437,437,437,437,437
Winter Weather,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,341,341,899,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,899,476,899,899,899,0,0,899,899,899,899,899


# Grab events to run
see reference: /home/csutter/DRIVE-clean/weather_events/notebooks/nws_dataset_analysis.ipynb for more info

Important notes and tracking 
- See "HERE!!" comments below, there are many now that we adjust logic depending on event
- DONE: Blizzard 
    - On 1/23 ran the remaining blizzard events from NCEI that hadnt already been ran in NWS warnings, using the same 5-min logic and the same 15 min buffer that I did from the NWS code. Noting this, bc we changed the code since (see bullet below). Note we have some cases (set35) where there are no images thus no model runs. 
- DONE: Lake-Effect Snow. 
    - For lake effect, severe snow, there are far too many events of too long length, to run ALL lake effect events for ALL years for ALL 5 mins during duration. Solution, run instances: 1) every 30 min and 2) by years at a time 
    - DONE: On 1/23, ran "Lake-Effect Snow", "2025", .dt.ceil("h"), .dt.floor("30min"), freq="30min"
        - (starting w/ 2025, logic here bc i didnt label images from 2025 so we have some justification for that.)
    - On 1/25: run Lake-Effect for 2024 w/ 30 min ceil and freq (made new savedout events_ofint dataset w 2024 and 2025 togethr)
- On 1/25, ran "Heavy Snow"
    - on 1/25: these events have an avg duration of 1.3 days... so running every 5 min not a great choice, chose to do every 30 min like for lakeeffect. 2025 only has 2 events. Extend to 2024 (don't have too much labeled data for that year, so that's decently consistent w the logic for lakeeffect above)
- On 1/25 generally choosing to prioritize gather more events over gathering more frequency within smaller number of events
    - If i want my code to run for 5 days, I can aim for 2000 instances per run...
    - EVERY 15 min for 2024/2025 -- DECIDED THIS FOR AMS WEEK. Started jobs 1/25 around 2pm. Will take 4 ish days to run them at most. 
        - Lake-effect every 15 min is 2000 - 4 days
        - Heavy snow every 15 min is 1000 - 2 days
        - Heavy rain is tiny - SKIP BUT NEED TO RUN ONCE ONE IS DONE! *
        - Blizzard is done 
        - Winter Storm is 2200 - 4-5 days
        - Winter Weather is 3300 - 6 ish days
    - EVERY 30 min (roughly half of ^...) 
        - Lake-effect every 30 min for 2024/2025 is 567 obs -- 28h
        - Heavy snow is 430 - 21h
        - Heavy rain is tiny - fast
        - Blizzard is done 
        - Winter Storm is 1000 - 2d
        - Winter Weather 1300 - 2.7d


    



In [65]:
# grab snow squall data (or whatever event of interest is)
d_event_subset = d_readin[d_readin["EVENT_TYPE"]=='Heavy Rain'] # HERE!!
print(len(d_event_subset))

#TOO MANY EVENTS! Limit it to 2023 and 2024
d_event_subset = d_event_subset[d_event_subset["YEAR"].isin([2024,2025])]  # HERE!!

# just for reference
print(len(d_event_subset))

print(np.mean(d_event_subset["duration"]))

14
12
0 days 03:47:10


In [55]:
print(len(d_event_subset))

458


In [66]:
# Grab start and end times, for which we'll run all datetimes in between
# Will grab start time, end time, and list every 5-min increment in between
# Will also run 15 mins before and after squall starts to capture the prior and post conditions
# Round to 0005, 0010, etc, 5 min increments. Why? B/c 1) we only snapshot images in 5-min increments anyway, and while they won't be exactly on the even 5 mins, if we consistently run inference runs with ever 5 mins, it means we'll capture all instances (e.g. if we started allowing "off" times like 11 min rather than 10, that may be the same "image instance" for 1000 cams -- this is a waste of inference run). This allows us to keep track seamlessly across lots of different case study inference runs, exactly which datetimes have and have not been ran already
# Note: we run every instance statewide, just for consistency/ease of run (see note above about seamless tracking across case studies). I'm not parsing certain regions based on those that had the squall in region. 

d_event_subset["start_round"] = d_event_subset["BEGIN_UTC"].dt.ceil("15min") #HERE!!
# For events where we want to limit to running 30 mins, to keep things organized, initiate this rounding to the nearest hour AFTER the event began
# prior version for 5 min: .dt.round("5min") # DEPRECATE this for consistency, should really always just ceil, and then rely on the "buffer" if want to capture safe surrounding times.
# for rounding to 30 min .dt.ceil("30min"), or .dt.ceil("5min")
d_event_subset["end_round"] = d_event_subset["END_UTC"].dt.floor("15min") # HERE!!
# To Round end time either top or bottom of hour that is less than the actual end time (bc want this last observation to actually capture the event duration, not just blind rounding which may lead to after)
# .dt.round("5min") # Similar to note above for ceil, will DEPRECATE this
# .dt.floor("30min")


# HERE!! Comment out -/+ if dont want a buffer
# DEPRECATING THE USE OF BUFFER TIMES. For now my thinking is that this only really matters for short lived squall events, and that data is only in NWS warnings and we did use 15 min buffer for that. 
d_event_subset["start_buffer"] = d_event_subset["start_round"] #$- pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] #+ pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="15min"),  # HERE!! Adjust if 5 min too frequent if an event is quite long
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)

print(len(d_event_subset))



12


In [43]:
print(len(d_event_subset))

136


In [67]:
# SAVE OUT THIS CLEANED DATASET FOR WHEN DOING ANALYSIS/MODELING
# IMPORTANT: may want to skip this step initially while you see how many unique dates will ened to be ran based on the logic (and may tweak the logic we set accordingly based on the number of expected runs)... so keep running the code after this cell to make sure the # of runs matches what we'd want to run before saving this out. 
# Set proper naming for the type of subsetting logic - must include 1) event name 2) any year subsetting 3) start time rounding and end time rounding 4) use of buffering 5) frequency of run within the duration of event
# E.g. "/blizzard_allyrs_ceilfloor5min_nobuffer_freq5min.csv", "lakeeffect_2025_ceilfloor30min_nobuffer_freq30min"
d_event_subset.to_csv("/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavyrain_2425_ceilfloor15min_nobuffer_freq15min.csv") #HERE!! 

# Important note: if you just click the CSV or even download the CSV and inspect in excel, it looks off... but if you read it in as a df and use it in a notebook, it reads in totally fine

In [68]:
# make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

datetimes_all = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        datetimes_all.append(j)

print(len(datetimes_all)) # this numnber should roughly match the # events x avg duration x numebr of runs per hour (e.g. 12 if running 5 min)
print(len(np.unique(datetimes_all)))

190
188


In [69]:
print(len(datetimes_all))
# Note that some will be overlapping for nearby counties or overlapping zones

dates_list = np.unique(datetimes_all)
print(len(dates_list)) # this is the amount of instances to run!

print(dates_list[0:4])

# Need to also see which datetimes I've already ran from past inference! See other notebook. 


190
188
['20240618_0900' '20240618_0915' '20240618_0930' '20240618_0945']


Grab list of datetimes already ran in inference  
[IMPORTANT REFERENCE CODE]
- Run this chunk and use pred_datetimes anytime we're making a new list of dates to run
- See here for the base reference code: /home/csutter/DRIVE-clean/operational_analysis/notebooks/summarize_dates_ran.ipynb

In [42]:
# grab all inf dirs:
#HERE!! update this list

ld = glob("/home/csutter/DRIVE-clean/operational_runs/*")
ld = sorted(ld)
# print(ld)

In [44]:
inf_sets_ran = ld

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


print(pred_datetimes[0:4])

15596
15596
['20250210_1000', '20250204_1000', '20250228_1000', '20250227_1000']


In [71]:
# Optional: if need to add in stuff that already IS running simulatneously so dont want to double on dates if they exist in multiple runs
# HERE!!

# OPT 1: if have 1 or 2 currently running jobs from which to remove dates running (then comment out OPT 2)

dates_runnING1 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/dates.csv")["date"])
print(len( dates_runnING1))
dates_runnING2 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set38_heavysnow/dates.csv")["date"])
dates_runnING3 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set39_winterstorm/dates.csv")["date"])
dates_runnING4 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set40_winterweather/dates.csv")["date"])
dates_runnING = dates_runnING1 + dates_runnING2 + dates_runnING3 + dates_runnING4
print(len(dates_runnING))

# OPT 2: if no currently running jobs to remove datetimes from list

# dates_runnING = []

2130
4942


In [72]:
# Cross check from dates_list and remove any dates already ran/running

dates_list_new = []
for d in dates_list:
    if d not in pred_datetimes:
        dates_list_new.append(d)

print("Unique datetimes from event of int")
print(len(dates_list))

print("Unique datetimes removing what's been ran")
print(len(dates_list_new))

dates_list_new2 = []
for d in dates_list_new:
    if d not in dates_runnING:
        dates_list_new2.append(d)

print("Unique datetimes removing what's currently running")
print(len(dates_list_new2))

Unique datetimes from event of int
188
Unique datetimes removing what's been ran
186
Unique datetimes removing what's currently running
186


In [64]:
# Save out 
forcsv = pd.DataFrame(dates_list_new2, columns=['date']) # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set41_heavyrain" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir, exist_ok=True)
forcsv.to_csv(f"{savetodir}/dates.csv") 

# Identify non-events

Try negative sampling first 
- See if there are dates that far between any events in the state
- Could do this at a more local scale (ie identify nonevents for Albany region, for example) but start simple by seeing if there are dates that arent associated with ANY event ANY location statewide. 

In [3]:
d_readin = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/ncei_events/ncei_ny_events_clean.gpkg")

d_readin.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")

# may take ~40 seconds

In [31]:
df = d_readin

In [32]:
# d_readin.head(4)
df.columns

Index(['Unnamed: 0', 'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',
       'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID',
       'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE',
       'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE',
       'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT',
       'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS',
       'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY',
       'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO',
       'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME',
       'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE',
       'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT',
       'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE',
       'CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT', 'BEGIN_UTC', 'END_UTC',
       'duration_sec', 'geometry', 'duration', 'ymd', 'ymd_be

In [33]:
df.head(3)

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,ymd,ymd_begin,ymd_end
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,36,2022,February,Winter Storm,Z,31,WESTERN CLINTON,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04
1,15,202202,3,900,202202,4,1600,164922,995749,NEW YORK,36,2022,February,Winter Storm,Z,30,SOUTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,030,030,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.98640 44.70781, -73.96619 44.709...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04
2,16,202202,3,900,202202,4,1600,164922,995750,NEW YORK,36,2022,February,Winter Storm,Z,27,NORTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 8 to 10 inches with...,CSV,027,027,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-74.66310 44.99891, -74.66100 44.999...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04


In [20]:
d_readin["BEGIN_DATE_TIME"][0].date()


datetime.date(2022, 2, 3)

In [119]:
# grab the date the event began and ended
df["ymd_begin"] = df["BEGIN_UTC"].dt.date
df["ymd_end"] = df["END_UTC"].dt.date

# add a 3 day buffer around it (e.g. severe snow events take time to clear)
df['ymd_begin_buffer'] = df['ymd_begin'] - pd.Timedelta(hours = 12) # Have a small buffer before the event, what matters more is the after event buffer
df['ymd_end_buffer'] = df['ymd_end'] + pd.Timedelta(days=3) # Allow 3 days to clear roads/event to settle. 

# add the range of dates between begin and ends
def generate_dates(row):
    return pd.date_range(start=row['ymd_begin_buffer'], end=row['ymd_end_buffer'], freq='D').date.tolist()

df['dates_between'] = df.apply(generate_dates, axis=1)


In [42]:
df.head(4)

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,ymd,ymd_begin,ymd_end,ymd_begin_buffer,ymd_end_buffer,dates_between
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,36,2022,February,Winter Storm,Z,31,WESTERN CLINTON,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-01-31,2022-02-07,"[2022-01-31, 2022-02-01, 2022-02-02, 2022-02-0..."
1,15,202202,3,900,202202,4,1600,164922,995749,NEW YORK,36,2022,February,Winter Storm,Z,30,SOUTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,030,030,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.98640 44.70781, -73.96619 44.709...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-01-31,2022-02-07,"[2022-01-31, 2022-02-01, 2022-02-02, 2022-02-0..."
2,16,202202,3,900,202202,4,1600,164922,995750,NEW YORK,36,2022,February,Winter Storm,Z,27,NORTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 8 to 10 inches with...,CSV,027,027,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-74.66310 44.99891, -74.66100 44.999...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-01-31,2022-02-07,"[2022-01-31, 2022-02-01, 2022-02-02, 2022-02-0..."
3,17,202202,3,900,202202,4,1600,164922,995752,NEW YORK,36,2022,February,Winter Storm,Z,29,SOUTHEASTERN ST. LAWRENCE,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 6 to 10 inches with...,CSV,029,029,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-75.06280 44.05041, -75.06920 44.053...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-01-31,2022-02-07,"[2022-01-31, 2022-02-01, 2022-02-02, 2022-02-0..."


In [115]:
list_dates_eventoccupied = []

for i in list(df["dates_between"]):
    for j in i:
        list_dates_eventoccupied.append(j)

print(len(list_dates_eventoccupied)) # before dropping dup dates

list_dates_eventoccupied = list(set(list_dates_eventoccupied))
print(len(list_dates_eventoccupied))

# Keep in datetime.date format for now because we have to compare to the windows of interest to find gap dates

34484
1012


In [121]:
# Find any gap dates that exist for winter 2024 and 2025

# 1. Define your study windows
windows = [

    # Decided to use this version of date ranges to find "nonevents". 
    # Note: I was trying to stay in 2024/2025 but needed to expand to other winter seasons (was trying to avoid that due to labeling in past years, but the benefits of having a broader range of dates outweighs the cons of having a small sample of labeled data from that season. Could also just exlude any datetimes from labeled dataset...)
    ('2022-12-01', '2023-02-28'), 
    ('2023-12-01', '2024-02-29'), 
    ('2024-12-01', '2025-02-28'), 

    # To match seasons with 2024/2025 events ran, use these three ranges
    # This logic, which is ideal b/c it matches with the ranges ran for events model runs.. but it only led to 12 dates, all between Jan 2024-Feb 2024. 
    # ('2024-01-01', '2024-02-29'), 
    # ('2024-12-01', '2025-02-28'), 
    # ('2025-12-01', '2025-02-28'),  # IMPORTANT note: this is because the NCEI storm events database for 2025 seems to cap at 10/25/25...
]

# 2. Build a "Master List" of every possible date
all_potential_dates = []
for start, end in windows:
    date_range = pd.date_range(start=start, end=end, freq='D').date.tolist()
    all_potential_dates.extend(date_range)

# 3. Filter out the occupied dates
# This says: "Keep the date ONLY if it is not in my occupied list"
non_event_dates = [d for d in all_potential_dates if d not in list_dates_eventoccupied]

# Result
print(f"Total possible days: {len(all_potential_dates)}")
print(f"Non-event days remaining: {len(non_event_dates)}")


# Convert final list of eligible gap dates to YYYYMMDD string

# List to store the formatted strings
eligibledates = []

# Iterate through the list and format each date
for date_obj in non_event_dates:
    # Use .strftime() to format the date
    formatted_date_str = date_obj.strftime("%Y%m%d")
    eligibledates.append(formatted_date_str)

# Print the resulting list of strings
print(len(eligibledates))

Total possible days: 271
Non-event days remaining: 53
53


In [122]:
print(eligibledates) # DECIDED TO USE THIS VERSION:  which applies a required buffer of 12 hours before events, 3 days after event buffer, there are 10 stretches and 53 unique dates

# alternative: there are only 5 "stretches" of dates, encompassing 27 total dates. But this is a good starting point given our logic for excluding dates was stringent. 

# alternative: if change the buffer to 2 days around the event start and end date, increases to 9 stretches and 43 dates

['20221207', '20221208', '20221209', '20221210', '20221231', '20230101', '20230102', '20230103', '20230104', '20230105', '20230106', '20230107', '20230108', '20230109', '20230110', '20230111', '20230112', '20230130', '20230131', '20230214', '20230219', '20230220', '20231208', '20231209', '20231210', '20231228', '20231229', '20231230', '20231231', '20240101', '20240102', '20240103', '20240104', '20240105', '20240202', '20240203', '20240204', '20240205', '20240206', '20240207', '20240208', '20240209', '20240210', '20240211', '20240212', '20240223', '20240224', '20240225', '20240226', '20240227', '20250223', '20250224', '20250225']


In [123]:
d_readin.head(3)

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,ymd,ymd_begin,ymd_end,ymd_begin_buffer,ymd_end_buffer,dates_between
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,36,2022,February,Winter Storm,Z,31,WESTERN CLINTON,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-02-03,2022-02-07,"[2022-02-03, 2022-02-04, 2022-02-05, 2022-02-0..."
1,15,202202,3,900,202202,4,1600,164922,995749,NEW YORK,36,2022,February,Winter Storm,Z,30,SOUTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,030,030,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.98640 44.70781, -73.96619 44.709...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-02-03,2022-02-07,"[2022-02-03, 2022-02-04, 2022-02-05, 2022-02-0..."
2,16,202202,3,900,202202,4,1600,164922,995750,NEW YORK,36,2022,February,Winter Storm,Z,27,NORTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 8 to 10 inches with...,CSV,027,027,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-74.66310 44.99891, -74.66100 44.999...",1 days 07:00:00,2022-02-03,2022-02-03,2022-02-04,2022-02-03,2022-02-07,"[2022-02-03, 2022-02-04, 2022-02-05, 2022-02-0..."


Prep list for running operational set. 
- Run every 2 hours (i.e. 12 times per day) starting at 0z, 2Z, .., for  each of the 53 dates = 636 obs
- Will run in 3 jobs simultaneously, 212 obs per job (i.e. ~11 hours per job)

In [131]:
datesrun = []
for d in eligibledates:
    for hh in ["0000","0200","0400","0600","0800","1000","1200","1400","1600","1800","2000","2200"]:
        datesrun.append(f"{d}_{hh}")

print(len(datesrun))

636


In [133]:
# Save out 
forcsv = pd.DataFrame(datesrun, columns=['date'])[0:200] # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir, exist_ok=True)
forcsv.to_csv(f"{savetodir}/dates.csv") 

In [ ]:
# NOTE! For saving out a cleaned dataframe of "non-events" as eventsofinterest, see this notebook: /home/csutter/DRIVE-clean/weather_events/notebooks/colocate_eventsregions_cams.ipynb

# Collect some datetimes before / after events 
- Noticed that I wanted to do this after analyzing the prediction counts of a blizzard event which had high proportion of snow_severe predictions, and was curious about what the ramp up/ drop off looked like
- To save run time for model runs, take advantage of the fact that I already ran events stats and can see which locations don't have cams (and thus wont have model preds) so we can save time by not pulling those dates that correspnd to locations that we already know dont have cams

In [3]:
from ast import literal_eval

# See /home/csutter/DRIVE-clean/weather_events/notebooks/visualize_stats_events_modelpred.ipynb for more code analyzing the model predictions by events

# Reading in these results comes from that notebook ^


#### Look at events and their model predictions when they have cameras associated with them (i.e. the model preds are meaningful)

stats = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/models/stats_events_modelpred/stats.csv")

# to read in the model counts col as dictionaries
stats["model_counts"] = stats["model_counts"].apply(literal_eval)

print("all rows (not cleaned)")
print(len(stats))

stats.head(4)

stats.columns

print(np.unique(stats["type"]))

stats_wpreds = stats[stats["model_counts"]!={}].reset_index()

print("rows WITH camera preds")
print(len(stats_wpreds))

all rows (not cleaned)
96003
['Blizzard' 'Heavy Rain' 'Heavy Snow' 'Lake-Effect Snow' 'Winter Storm'
 'Winter Weather']
rows WITH camera preds
58679


In [4]:
print(len(np.unique(stats_wpreds["event"])))

471


In [6]:
stats_wpreds.head(4)

,index,event,episode,type,location,wfo,timestamp,model_counts
0,0,996866,165058,Blizzard,SOUTHWEST SUFFOLK,OKX,20220129_1200,"{'poor_viz': 5, 'snow_severe': 54}"
1,1,996866,165058,Blizzard,SOUTHWEST SUFFOLK,OKX,20220129_1800,"{'snow': 1, 'snow_severe': 58}"
2,2,996866,165058,Blizzard,SOUTHWEST SUFFOLK,OKX,20220129_1300,"{'poor_viz': 5, 'snow': 1, 'snow_severe': 53}"
3,3,996866,165058,Blizzard,SOUTHWEST SUFFOLK,OKX,20220129_1600,"{'poor_viz': 1, 'snow_severe': 58}"


In [26]:
# grab 30 min before and 30 min after every event in that df

# for each event, grab the first timestamp and the last timestamp and take every 15 min timestep leading up to the hour before/after

buffertimes_allevents = []

for ev in np.unique(stats_wpreds["event"]):
    # grab the start and end time strings (eg 20220129_1200)
    evdf = stats_wpreds[stats_wpreds["event"]==ev]
    # display(evdf.head(4))
    sortdf = evdf.sort_values("timestamp").reset_index()
    # display(sortdf.tail(4))
    # print(len(sortdf))
    first = sortdf["timestamp"][0]
    last = sortdf["timestamp"][len(sortdf)-1]
    
    # convert to datetime format so we can use timedelta to make things easier
    # then grab the time corresponding to 60 min before / after the start / end
    firstdt = pd.to_datetime(first, format="%Y%m%d_%H%M") 
    firstbuffer = firstdt - pd.Timedelta(minutes=60)
    lastdt = pd.to_datetime(last, format="%Y%m%d_%H%M") 
    lastbuffer = lastdt + pd.Timedelta(minutes=60)

    # grab list of 15 min increments for an hour before/after
    beforestorm = pd.date_range(start=firstbuffer, end=firstdt, freq="15min")
    # drop the "firstdt" which is the storm start time which was already ran
    beforestorm = beforestorm[:-1]
    
    afterstorm = pd.date_range(lastdt, lastbuffer, freq="15min")
    # drop the "lastdt" which is the storm end time which was already ran
    afterstorm = afterstorm[1:]

    # convert back to format of string yyyymmdd_hhmm
    beforestrings = [d.strftime("%Y%m%d_%H%M") for d in beforestorm]
    afterstrings = [d.strftime("%Y%m%d_%H%M") for d in afterstorm]

    # append all times for before/after buffer to list
    for d in beforestrings:
        buffertimes_allevents.append(d)
    for d in afterstrings:
        buffertimes_allevents.append(d)



# d_event_subset["start_buffer"] = d_event_subset["start_round"] #$- pd.Timedelta(minutes=15)
# d_event_subset["end_buffer"] = d_event_subset["end_round"] #+ pd.Timedelta(minutes=15)

# d_event_subset["all_times"] = d_event_subset.apply(
#     lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="15min"),  # HERE!! Adjust if 5 min too frequent if an event is quite long
#     axis=1
# )

# d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
#     lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
# )

# d_event_subset.head(4)

# print(len(d_event_subset))



# stats_wpreds[0:4]

In [23]:
print(buffertimes_allevents[0:12])

['20220129_1100', '20220129_1115', '20220129_1130', '20220129_1145', '20220129_1845', '20220129_1900', '20220129_1915', '20220129_1930', '20220129_1100', '20220129_1115', '20220129_1130', '20220129_1145']


In [28]:
# Quick check before dropping dups and saving out dates to do runs for
# w/ 471 events, should have 8 new datetimes to pull for each one (4 before, 4 after)
print(471*8)
print(len(buffertimes_allevents))
# 

3768
3768


In [30]:
# Drop any dup dates contained within the list
buffertimes = np.unique(buffertimes_allevents)
print(len(buffertimes))

1279


In [45]:
# Then the rest - check for any date times that have already been ran in inferenc_runs
# NOTE: needed to have ran cells above [IMPORTANT REFERENCE CODE] to get this list -- pred_datetimes

dates_list_new = []
for d in buffertimes:
    if d not in pred_datetimes:
        dates_list_new.append(d)

print(len(dates_list_new))

279


In [ ]:
# Save out 
forcsv = pd.DataFrame(dates_list_new, columns=['date']) # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set45_buffermisc" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir, exist_ok=True)
forcsv.to_csv(f"{savetodir}/dates.csv") 